# E0 — Baseline LSTM

**E0 — minimal LSTM, random embeddings, no regularization. Reference point for every other experiment.**

In [2]:
import os, re, time, json, pickle
import numpy as np
import pandas as pd

SEED = 42
DATA_PATH = "../data/IMDB Dataset.csv"
RESULTS_DIR = "../results"
TOKENIZER_PATH = "../results/tokenizer.pkl"
VOCAB_SIZE = 10000
EMBED_DIM = 100
SAMPLE_SIZE = 15000   # <-- subsample for faster training

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["review", "sentiment"])
df["review"] = df["review"].apply(lambda t: re.sub(r"<br\s*/?>", " ", str(t)))
df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})
assert df["label"].isna().sum() == 0, "Unexpected sentiment values — check the column."

# Subsample BEFORE splitting, so train/test shrink together and stay balanced
df = df.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)

X = df["review"].astype(str).to_numpy()
y = df["label"].to_numpy(dtype=int)

from sklearn.model_selection import train_test_split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED,
)
print(f"Train: {len(X_train_text)}  Test: {len(X_test_text)}")

Train: 12000  Test: 3000


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

tf.random.set_seed(SEED)
np.random.seed(SEED)
MAX_LEN = 200
BATCH_SIZE = 64
EPOCHS = 5

I0000 00:00:1788181814.188954   90150 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788181814.814384   90150 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788181816.680255   90150 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Reuse the SAME tokenizer across every notebook (loads from disk if a
# previous notebook already built one) so all experiments share one vocab —
# that's what makes comparing accuracy across them fair.
if os.path.exists(TOKENIZER_PATH):
    with open(TOKENIZER_PATH, "rb") as f:
        tokenizer = pickle.load(f)
    print("Loaded existing tokenizer.")
else:
    tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
    tokenizer.fit_on_texts(X_train_text)
    os.makedirs(RESULTS_DIR, exist_ok=True)
    with open(TOKENIZER_PATH, "wb") as f:
        pickle.dump(tokenizer, f)
    print("Built and saved new tokenizer.")

Loaded existing tokenizer.


In [5]:
x_train = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=MAX_LEN)
x_test = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=MAX_LEN)

model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    LSTM(64),
    Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

/home/manikya/nlp_project/venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1788181822.820567   90150 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3536 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
start = time.time()
history = model.fit(x_train, y_train, validation_split=0.1,
                     batch_size=BATCH_SIZE, epochs=EPOCHS, verbose=2)
train_time = time.time() - start

Epoch 1/5


I0000 00:00:1788181825.633391   90316 cuda_dnn.cc:461] Loaded cuDNN version 92400


169/169 - 8s - 46ms/step - accuracy: 0.7455 - loss: 0.5089 - val_accuracy: 0.8325 - val_loss: 0.3845
Epoch 2/5
169/169 - 5s - 27ms/step - accuracy: 0.8817 - loss: 0.2898 - val_accuracy: 0.8700 - val_loss: 0.3420
Epoch 3/5
169/169 - 5s - 29ms/step - accuracy: 0.9208 - loss: 0.2068 - val_accuracy: 0.8500 - val_loss: 0.3981
Epoch 4/5
169/169 - 4s - 26ms/step - accuracy: 0.9319 - loss: 0.1782 - val_accuracy: 0.8467 - val_loss: 0.4219
Epoch 5/5
169/169 - 5s - 27ms/step - accuracy: 0.9506 - loss: 0.1336 - val_accuracy: 0.8375 - val_loss: 0.5441


In [7]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

probs = model.predict(x_test, batch_size=BATCH_SIZE).ravel()
preds = (probs > 0.5).astype(int)

acc = accuracy_score(y_test, preds)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average="binary")
auc = roc_auc_score(y_test, probs)

row = {
    "run_name": "E0_baseline_lstm",
    "accuracy": round(acc, 4), "precision": round(prec, 4), "recall": round(rec, 4),
    "f1": round(f1, 4), "roc_auc": round(auc, 4),
    "train_time_sec": round(train_time, 1),
    "epochs_run": len(history.history["loss"]),
    "params": model.count_params(),
    "max_len": MAX_LEN, "embeddings": "random_init",
}

os.makedirs(RESULTS_DIR, exist_ok=True)
out_path = os.path.join(RESULTS_DIR, "E0_baseline_lstm.csv")
pd.DataFrame([row]).to_csv(out_path, index=False)
print(f"Saved {out_path}")
print(json.dumps(row, indent=2))

47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
Saved ../results/E0_baseline_lstm.csv
{
  "run_name": "E0_baseline_lstm",
  "accuracy": 0.8333,
  "precision": 0.7809,
  "recall": 0.9321,
  "f1": 0.8498,
  "roc_auc": 0.9193,
  "train_time_sec": 26.2,
  "epochs_run": 5,
  "params": 1042305,
  "max_len": 200,
  "embeddings": "random_init"
}
